# Assignment 3 - Border / padding, and image complement

> 1. WAP to draw a border around the input image (in other words, pad the input image).
>    - Input the width of the border (in pixel count) and the desired colour of the border from the user.
>    - Try this for both binary and grayscale images.
> 2. WAP for image complement. Try this for both binary and grayscale images.

### What you actually learn here
- creating arrays from scratch (`np.zeros`, `np.full`) and *placing* an image inside one
- slice assignment - how you write into a region of an array
- why padding exists at all (every convolution in the rest of the course needs it)
- signed arithmetic on `uint8`, again, from the other direction
- writing one function that handles 2D and 3D inputs without duplicating itself

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

gray = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)
rgb  = cv2.cvtColor(cv2.imread("images/lena.jpg"), cv2.COLOR_BGR2RGB)
binr = to_bin = ((cv2.imread("images/sudoku.png", cv2.IMREAD_GRAYSCALE) > 128) * 255).astype(np.uint8)
_imgs, _titles = [rgb, gray, binr], ["colour", "grayscale", "binary"]
fig, axes = plt.subplots(1, 3, figsize=(3.5 * 3, 3.5), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
print("shapes:", rgb.shape, gray.shape, binr.shape)

## Slice assignment is the whole trick

To put a `(10, 12)` image in the middle of a `(16, 18)` canvas with a 3-pixel border:

```python
canvas = np.full((16, 18), 200, dtype=np.uint8)   # fill with the border colour
canvas[3:13, 3:15] = img                          # overwrite the interior
```

Note `3:13` is `3 : 3 + 10`. Writing `3:-3` also works and reads better. Both are fine;
know that they are the same thing.

### The 2D-vs-3D wrinkle

A grayscale border colour is one number (`200`). A colour border is three (`(255, 0, 0)`).
So the canvas you allocate has a different shape in each case. `np.full` will do the right
thing if you hand it the right shape and value, so branch on `img.ndim` once and keep the
rest of the function shared.

In [ ]:
# Slice assignment, small enough to read every number.
canvas = np.full((7, 9), 9, dtype=np.uint8)
canvas[2:5, 2:7] = 1
print(canvas)
print("\ninterior shape:", canvas[2:5, 2:7].shape, " == ", canvas[2:-2, 2:-2].shape)

---
## Task 1 - add a border

`width=0` must return the image unchanged. Your tests also compare the result against
`cv2.copyMakeBorder`, so get the geometry exactly right: the output grows by `2*width` on
**height and width only** - a colour image keeps its 3 channels.

In [ ]:
def my_copyMakeBorder(img, width, color):
    """Surround an image with a solid border.

    Parameters
    ----------
    img   : (H, W) or (H, W, 3) uint8 array
    width : int >= 0, border thickness in pixels, same on all four sides
    color : int for a 2D image, or a (r, g, b) tuple for a 3-channel image

    Returns
    -------
    uint8 array of shape (H + 2*width, W + 2*width) or (H + 2*width, W + 2*width, 3).
    The original pixels are unchanged and centred; everything else is `color`.
    width=0 returns an unchanged copy.
    """
    # TODO: allocate a canvas filled with `color`, then drop `img` into the middle
    raise NotImplementedError

### Test `my_copyMakeBorder` yourself

- grayscale: a `(10, 12)` image of 50s, `width=3`, `color=200` → uint8, shape `(16, 18)`,
  the original sits untouched at `[3:13, 3:15]`, and all four outer edges are 200
- `width=0` returns the image unchanged
- colour: lena, `width=5`, `(255,0,0)` → shape `(H+10, W+10, 3)`, the corner pixel is
  `(255,0,0)`, the interior equals the original
- identical to `cv2.copyMakeBorder(rgb, 7, 7, 7, 7, cv2.BORDER_CONSTANT, value=(0,255,0))`
- binary: sudoku thresholded at 128, bordered with `width=4`, `color=255` → still only 0 and 255

In [ ]:
# Your tests for my_copyMakeBorder: one assert per property above.


In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.copyMakeBorder(src, top, bottom, left, right, borderType, value=None)
# Note it takes FOUR separate widths, not one -- asymmetric borders come free.
import cv2, numpy as np
cv_out = cv2.copyMakeBorder(rgb, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=(255, 0, 0))
assert np.array_equal(cv_out, my_copyMakeBorder(rgb, 20, (255, 0, 0))), "disagrees with cv2"
print("match. borderType values:")
print("  BORDER_CONSTANT (needs value=), BORDER_REPLICATE, BORDER_REFLECT,")
print("  BORDER_REFLECT_101 (the default in most filters), BORDER_WRAP")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Shape first: `h, w = img.shape[:2]`, then the output shape is
`(h + 2*width, w + 2*width)` plus `(3,)` if `img.ndim == 3`.

`np.full(shape, color, dtype=np.uint8)` accepts a 3-tuple `color` when the last
axis of `shape` is 3 - broadcasting again. Then one slice assignment.

</details>

In [ ]:
_imgs, _titles = [my_copyMakeBorder(gray, 20, 0), my_copyMakeBorder(gray, 20, 255), my_copyMakeBorder(rgb, 20, (255, 0, 0)), my_copyMakeBorder(binr, 12, 255)], ["gray, black border", "gray, white border", "colour, red border", "binary, white border"]
fig, axes = plt.subplots(1, 4, figsize=(3.2 * 4, 3.2), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
print("with border:", my_copyMakeBorder(rgb, 20, (255,0,0)).shape, "was", rgb.shape)

### Why anybody cares about padding

Every filter in the next few weeks slides a `k x k` window over the image. At the edge the
window hangs off. Padding is how you decide what it sees there, and the choice changes your output:

In [ ]:
tiny = gray[:64, :64]
modes = {"constant 0": cv2.BORDER_CONSTANT, "replicate": cv2.BORDER_REPLICATE,
         "reflect": cv2.BORDER_REFLECT, "wrap": cv2.BORDER_WRAP}
_imgs, _titles = list([cv2.copyMakeBorder(tiny, 24, 24, 24, 24, m) for m in modes.values()]), list(modes)
fig, axes = plt.subplots(1, len(_imgs), figsize=(3.0 * len(_imgs), 3.0), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

Your `my_copyMakeBorder` implements the first one, `BORDER_CONSTANT`. Now you know what the other
names in the OpenCV docs mean.

---
## Task 2 - image complement (the photographic negative)

For an 8-bit image the complement of intensity `r` is `255 - r`. Must work on grayscale,
colour, and binary, and applying it twice must return the original exactly.

In [ ]:
def my_bitwise_not(img):
    """Photographic negative of an 8-bit image.

    Parameters
    ----------
    img : uint8 array, any shape (2D grayscale, 3D colour, or binary 0/255)

    Returns
    -------
    uint8 array of the same shape where every value v becomes 255 - v.
    my_bitwise_not(my_bitwise_not(img)) must equal img.
    """
    # TODO: your code here
    raise NotImplementedError

### Test `my_bitwise_not` yourself

Fixture: the **ramp**: an `(8, 256)` uint8 image whose every row is `0, 1, ..., 255` (mean exactly 127.5).

- uint8, same shape
- 0 → 255, 255 → 0, 127 → 128
- applying it twice returns the original exactly
- along a ramp row the output never increases
- colour lena: identical to `cv2.bitwise_not` (each channel inverted independently)
- binary `[[0,255],[255,0]]` → `[[255,0],[0,255]]`

In [ ]:
# Your tests for my_bitwise_not: one assert per property above.


In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.bitwise_not(src) -- and it is NOT limited to binary images despite the name.
import cv2, numpy as np
assert np.array_equal(cv2.bitwise_not(rgb), my_bitwise_not(rgb)), "disagrees with cv2"
assert np.array_equal(cv2.bitwise_not(gray), my_bitwise_not(gray))
print("match. The bitwise family, all of which take an optional mask=:")
print("  cv2.bitwise_not / bitwise_and / bitwise_or / bitwise_xor")
print("  cv2.subtract(255, img) also works and SATURATES instead of wrapping")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Almost a one-liner, but think about the dtype for a second. `255 - img` where
`img` is `uint8`: is the intermediate result safe, or should you widen first?
Try `255 - np.uint8(0)` in a scratch cell and look at the dtype of the answer.

</details>

In [ ]:
_imgs, _titles = [gray, my_bitwise_not(gray), rgb, my_bitwise_not(rgb), binr, my_bitwise_not(binr)], ["gray", "negative", "colour", "negative", "binary", "negative"]
_cols = 6; _rows = -(-len(_imgs) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(3.0 * _cols, 3.0 * _rows), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
print("involution holds:", np.array_equal(my_bitwise_not(my_bitwise_not(rgb)), rgb))

### Where the negative is genuinely useful

X-rays and mammograms: small bright lesions on a dark field are easier for the eye to pick out
as dark spots on a bright field. Try it on the darkest image in the folder.

In [ ]:
import cv2
b = cv2.imread("images/board.jpg", cv2.IMREAD_GRAYSCALE)
_imgs, _titles = [b, my_bitwise_not(b)], [f"board.jpg (mean {b.mean():.0f})", "my_bitwise_not"]
fig, axes = plt.subplots(1, 2, figsize=(4 * 2, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 3))
plt.hist(b.ravel(), bins=256, range=(0, 256), color="steelblue")
plt.title("original"); plt.xlabel("intensity"); plt.ylabel("pixel count")
plt.xlim(0, 255); plt.tight_layout(); plt.show()

---
## Think about it

1. `my_bitwise_not` reversed the histogram. Describe the transformation of the histogram in words -
   is it a shift, a flip, a scale?
2. You bordered a binary image with `255`. What if a user asks for border colour `128` on a
   binary image? Your function will happily do it. Is the result still a binary image?
   Should the function refuse? Argue both sides in two sentences.
3. `my_copyMakeBorder(img, 20, 0)` and `cv2.copyMakeBorder(img, 20,20,20,20, cv2.BORDER_CONSTANT, value=0)`
   agree. Time both with `%timeit`. Which is faster, and does the gap matter at 1080p?
4. Compose the two: is `my_bitwise_not(my_copyMakeBorder(img, w, c))` the same as
   `my_copyMakeBorder(complement(img), w, complement_of_c)`? Verify with code.

In [ ]:
# Scratch space.

---
## Stretch

1. **Asymmetric borders.** Extend to `my_copyMakeBorder(img, top, bottom, left, right, color)`.
   Then use it to letterbox any image to a 16:9 canvas without distorting it.
2. **`np.pad`.** numpy already does this: `np.pad(img, 20, constant_values=200)`. Reproduce
   your Task 1 with it, including the colour case (harder than it looks - what does
   `pad_width` mean for a 3-channel array?).
3. **A framed collage.** Border each of the ten images in `images/` with a 10px white edge,
   resize them to a common size, and `np.hstack`/`np.vstack` them into one contact sheet.
   Save it with `cv2.imwrite` - remember to convert RGB back to BGR first.

In [ ]:
# Stretch scratch space.

---
## OpenCV API card - borders, arithmetic, bitwise

```python
dst = cv2.copyMakeBorder(src, top, bottom, left, right, borderType, value=None)
```
Four independent widths. `value` only matters for `BORDER_CONSTANT`.

| borderType | what the outside looks like, for `abcd|` |
|---|---|
| `BORDER_CONSTANT` | `vvvv|abcd|vvvv` |
| `BORDER_REPLICATE` | `aaaa|abcd|dddd` |
| `BORDER_REFLECT` | `dcba|abcd|dcba` |
| `BORDER_REFLECT_101` | `dcb|abcd|cba` — the **default** inside filters |
| `BORDER_WRAP` | `abcd|abcd|abcd` |

```python
dst = cv2.bitwise_not(src, mask=None)                # 255 - src
dst = cv2.bitwise_and(src1, src2, mask=None)         # also or / xor
dst = cv2.add(src1, src2)                            # SATURATES, never wraps
dst = cv2.subtract(src1, src2)
dst = cv2.addWeighted(src1, alpha, src2, beta, gamma)
dst = cv2.absdiff(src1, src2)
dst = cv2.convertScaleAbs(src, alpha=1, beta=0)      # |alpha*src + beta| -> uint8
```
Prefer these over `+`/`-` on `uint8` arrays: they saturate at 0 and 255 instead of wrapping.